## 1 — Imports & load clean data

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import pickle
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("./data/processed/hotel_bookings_clean.csv")
print(f"Loaded: {df.shape}")
print(df["is_canceled"].value_counts(normalize=True))

Loaded: (86372, 30)
is_canceled
0    0.725293
1    0.274707
Name: proportion, dtype: float64


## 2 — Australian public holidays & school holidays

In [2]:
# ── Australian Public Holidays (national + state-level key dates) ──────────
AU_PUBLIC_HOLIDAYS = {
    # (month, day): name
    (1, 1):   "New Year's Day",
    (1, 26):  "Australia Day",
    (4, 25):  "ANZAC Day",
    (12, 25): "Christmas Day",
    (12, 26): "Boxing Day",
}

# Melbourne Cup — first Tuesday of November (VIC)
# We'll flag November week 1 as a proxy
MELBOURNE_CUP_MONTH = 11
MELBOURNE_CUP_WEEK_APPROX = (1, 7)  # day range

def is_public_holiday(month, day):
    return int((month, day) in AU_PUBLIC_HOLIDAYS)

def is_melbourne_cup_week(month, day):
    return int(month == MELBOURNE_CUP_MONTH and MELBOURNE_CUP_WEEK_APPROX[0] <= day <= MELBOURNE_CUP_WEEK_APPROX[1])

# ── Australian School Holidays (approximate national overlap periods) ──────
# These are the periods where ALL states have holidays simultaneously
AU_SCHOOL_HOLIDAY_PERIODS = [
    # (month_start, day_start, month_end, day_end)
    (1, 1,  1, 31),   # January summer holidays
    (4, 1,  4, 21),   # Easter/Autumn break
    (7, 1,  7, 21),   # Winter break
    (9, 20, 10, 7),   # Spring break
    (12, 10, 12, 31), # Summer break starts
]

def is_school_holiday(month, day):
    for (ms, ds, me, de) in AU_SCHOOL_HOLIDAY_PERIODS:
        start = pd.Timestamp(2000, ms, ds)
        end   = pd.Timestamp(2000, me, de)
        check = pd.Timestamp(2000, month, day)
        if start <= check <= end:
            return 1
    return 0

# ── Major Australian Events ────────────────────────────────────────────────
# F1 Grand Prix Melbourne — ~March week 2
# Vivid Sydney — late May to mid June
# AFL Grand Final — late September
AU_MAJOR_EVENTS = [
    (3, 10, 3, 17),   # F1 Grand Prix Melbourne
    (5, 24, 6, 15),   # Vivid Sydney
    (9, 20, 9, 30),   # AFL Grand Final week
]

def is_major_event(month, day):
    for (ms, ds, me, de) in AU_MAJOR_EVENTS:
        start = pd.Timestamp(2000, ms, ds)
        end   = pd.Timestamp(2000, me, de)
        check = pd.Timestamp(2000, month, day)
        if start <= check <= end:
            return 1
    return 0

# Apply
df["is_public_holiday"]    = df.apply(lambda r: is_public_holiday(r["arrival_date_month_num"] if "arrival_date_month_num" in df.columns else pd.to_datetime(r["arrival_date_month"], format="%B").month, r["arrival_date_day_of_month"]), axis=1)

# Easier — build a month number column first
month_map = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
             "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
df["month_num"] = df["arrival_date_month"].map(month_map)

df["is_public_holiday"] = df.apply(
    lambda r: is_public_holiday(r["month_num"], r["arrival_date_day_of_month"]), axis=1)
df["is_school_holiday"]  = df.apply(
    lambda r: is_school_holiday(r["month_num"], r["arrival_date_day_of_month"]), axis=1)
df["is_major_event"]     = df.apply(
    lambda r: is_major_event(r["month_num"], r["arrival_date_day_of_month"]), axis=1)
df["is_melbourne_cup"]   = df.apply(
    lambda r: is_melbourne_cup_week(r["month_num"], r["arrival_date_day_of_month"]), axis=1)

print("Holiday/event flags added:")
print(df[["is_public_holiday","is_school_holiday","is_major_event","is_melbourne_cup"]].sum())

Holiday/event flags added:
is_public_holiday      833
is_school_holiday    24047
is_major_event       10110
is_melbourne_cup      1234
dtype: int64


## 3 — Date & season features

In [3]:
# Southern Hemisphere seasons
def au_season(month):
    if month in [12, 1, 2]:  return "Summer"
    elif month in [3, 4, 5]: return "Autumn"
    elif month in [6, 7, 8]: return "Winter"
    else:                     return "Spring"

df["au_season"] = df["month_num"].apply(au_season)

# Is weekend arrival
df["is_weekend_arrival"] = df["arrival_date_day_of_month"].apply(
    lambda d: 1 if d % 7 in [0, 6] else 0  # proxy — proper fix needs full date
)

# Build full arrival date
df["arrival_date"] = pd.to_datetime(
    df["arrival_date_year"].astype(str) + "-" +
    df["month_num"].astype(str) + "-" +
    df["arrival_date_day_of_month"].astype(str),
    errors="coerce"
)
df["day_of_week"]       = df["arrival_date"].dt.dayofweek   # 0=Mon, 6=Sun
df["is_weekend_arrival"]= df["day_of_week"].isin([4,5,6]).astype(int)  # Fri/Sat/Sun
df["quarter"]           = df["arrival_date"].dt.quarter

print("Date features added")
print(df[["arrival_date","au_season","day_of_week","is_weekend_arrival","quarter"]].head(5))

Date features added
  arrival_date au_season  day_of_week  is_weekend_arrival  quarter
0   2015-07-01    Winter            2                   0        3
1   2015-07-01    Winter            2                   0        3
2   2015-07-01    Winter            2                   0        3
3   2015-07-01    Winter            2                   0        3
4   2015-07-01    Winter            2                   0        3


## 4 — Booking behaviour features

In [4]:
# Total guests
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

# Total stay value proxy (ADR × nights)
df["total_stay_value"] = df["adr"] * df["total_nights"]

# Has any special request
df["has_special_request"] = (df["total_of_special_requests"] > 0).astype(int)

# Requires parking
df["needs_parking"] = (df["required_car_parking_spaces"] > 0).astype(int)

# Room was changed (reserved != assigned)
df["room_was_changed"] = (
    df["reserved_room_type"] != df["assigned_room_type"]
).astype(int)

# Guest has cancellation history
df["has_cancel_history"] = (df["previous_cancellations"] > 0).astype(int)

# Lead time buckets
df["lead_time_bucket"] = pd.cut(
    df["lead_time"],
    bins=[0, 7, 30, 90, 180, 365, 9999],
    labels=["0-7d", "8-30d", "31-90d", "91-180d", "181-365d", "365d+"]
)

# ADR bucket (AUD price tiers — proxy)
df["adr_tier"] = pd.cut(
    df["adr"],
    bins=[0, 60, 120, 200, 300, 9999],
    labels=["Budget", "Midrange", "Upscale", "Luxury", "Ultra"]
)

print("Behaviour features added:")
new_cols = ["total_guests","total_stay_value","has_special_request",
            "needs_parking","room_was_changed","has_cancel_history",
            "lead_time_bucket","adr_tier"]
print(df[new_cols].head(5))

Behaviour features added:
   total_guests  total_stay_value  has_special_request  needs_parking  \
0             1              75.0                    0              0   
1             1              75.0                    0              0   
2             2             196.0                    1              0   
3             2             214.0                    0              0   
4             2             206.0                    1              0   

   room_was_changed  has_cancel_history lead_time_bucket  adr_tier  
0                 1                   0             0-7d  Midrange  
1                 0                   0            8-30d  Midrange  
2                 0                   0            8-30d  Midrange  
3                 0                   0              NaN  Midrange  
4                 0                   0            8-30d  Midrange  


## 5 — Encode categoricals

In [5]:
from sklearn.preprocessing import OrdinalEncoder

# Columns to one-hot encode (low cardinality)
ohe_cols = ["hotel", "meal", "market_segment", "distribution_channel",
            "deposit_type", "customer_type", "au_season"]

# Columns to label encode (ordinal or high cardinality)
label_cols = ["reserved_room_type", "assigned_room_type",
              "lead_time_bucket", "adr_tier"]

# Drop columns not needed for modelling
drop_cols = ["arrival_date", "arrival_date_month", "country",
             "agent", "month_num"]

df_model = df.drop(columns=drop_cols)

# One-hot encode
df_model = pd.get_dummies(df_model, columns=ohe_cols, drop_first=True)

# Label encode ordinal
le = LabelEncoder()
for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print(f"Model-ready shape: {df_model.shape}")
print(df_model.dtypes.value_counts())

Model-ready shape: (86372, 60)
int64      32
bool       24
float64     2
int32       2
Name: count, dtype: int64


## 6 — Final feature set & save

In [8]:
# Separate features and target
TARGET = "is_canceled"
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print(f"Features: {X.shape[1]}")
print(f"Target balance:\n{y.value_counts(normalize=True)}")

# Save
X.to_csv("./data/processed/X.csv", index=False)
y.to_csv("./data/processed/y.csv", index=False)

# Save feature names for later
import json
with open("./data/processed/feature_names.json", "w") as f:
    json.dump(list(X.columns), f, indent=2)

print("\nSaved X.csv, y.csv, feature_names.json to data/processed/")
print("\nFinal feature list:")
for i, col in enumerate(X.columns):
    print(f"  {i+1:02d}. {col}")

Features: 59
Target balance:
is_canceled
0    0.725293
1    0.274707
Name: proportion, dtype: float64

Saved X.csv, y.csv, feature_names.json to data/processed/

Final feature list:
  01. lead_time
  02. arrival_date_year
  03. arrival_date_week_number
  04. arrival_date_day_of_month
  05. stays_in_weekend_nights
  06. stays_in_week_nights
  07. adults
  08. children
  09. babies
  10. is_repeated_guest
  11. previous_cancellations
  12. previous_bookings_not_canceled
  13. reserved_room_type
  14. assigned_room_type
  15. booking_changes
  16. days_in_waiting_list
  17. adr
  18. required_car_parking_spaces
  19. total_of_special_requests
  20. total_nights
  21. is_public_holiday
  22. is_school_holiday
  23. is_major_event
  24. is_melbourne_cup
  25. is_weekend_arrival
  26. day_of_week
  27. quarter
  28. total_guests
  29. total_stay_value
  30. has_special_request
  31. needs_parking
  32. room_was_changed
  33. has_cancel_history
  34. lead_time_bucket
  35. adr_tier
  36. hote

## 7 — Save as reusable module

In [9]:
src_code = '''import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import json

MONTH_MAP = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
             "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}

AU_PUBLIC_HOLIDAYS = {(1,1),(1,26),(4,25),(12,25),(12,26)}

AU_SCHOOL_HOLIDAY_PERIODS = [
    (1,1,1,31),(4,1,4,21),(7,1,7,21),(9,20,10,7),(12,10,12,31)
]

AU_MAJOR_EVENTS = [(3,10,3,17),(5,24,6,15),(9,20,9,30)]

def is_public_holiday(m, d):
    return int((m, d) in AU_PUBLIC_HOLIDAYS)

def is_school_holiday(m, d):
    for ms,ds,me,de in AU_SCHOOL_HOLIDAY_PERIODS:
        if pd.Timestamp(2000,ms,ds) <= pd.Timestamp(2000,m,d) <= pd.Timestamp(2000,me,de):
            return 1
    return 0

def is_major_event(m, d):
    for ms,ds,me,de in AU_MAJOR_EVENTS:
        if pd.Timestamp(2000,ms,ds) <= pd.Timestamp(2000,m,d) <= pd.Timestamp(2000,me,de):
            return 1
    return 0

def au_season(month):
    if month in [12,1,2]:   return "Summer"
    elif month in [3,4,5]:  return "Autumn"
    elif month in [6,7,8]:  return "Winter"
    else:                   return "Spring"

def build_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # Drop leakage & high-null
    drop = ["reservation_status","reservation_status_date","company"]
    df.drop(columns=[c for c in drop if c in df.columns], inplace=True)

    # Nulls
    df["agent"].fillna(0, inplace=True)
    df["children"].fillna(0, inplace=True)
    df["country"].fillna("Unknown", inplace=True)
    df["agent"] = df["agent"].astype(int)
    df["children"] = df["children"].astype(int)

    # Filter
    df = df[df["adr"] >= 0]
    df = df[(df["adults"] + df["children"] + df["babies"]) > 0]
    df.drop_duplicates(inplace=True)

    # Date
    df["month_num"] = df["arrival_date_month"].map(MONTH_MAP)
    df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    df = df[df["total_nights"] > 0]

    df["arrival_date"] = pd.to_datetime(
        df["arrival_date_year"].astype(str) + "-" +
        df["month_num"].astype(str) + "-" +
        df["arrival_date_day_of_month"].astype(str), errors="coerce")
    df["day_of_week"]        = df["arrival_date"].dt.dayofweek
    df["is_weekend_arrival"] = df["day_of_week"].isin([4,5,6]).astype(int)
    df["quarter"]            = df["arrival_date"].dt.quarter

    # AU flags
    df["is_public_holiday"] = df.apply(lambda r: is_public_holiday(r["month_num"], r["arrival_date_day_of_month"]), axis=1)
    df["is_school_holiday"]  = df.apply(lambda r: is_school_holiday(r["month_num"], r["arrival_date_day_of_month"]), axis=1)
    df["is_major_event"]     = df.apply(lambda r: is_major_event(r["month_num"], r["arrival_date_day_of_month"]), axis=1)
    df["au_season"]          = df["month_num"].apply(au_season)

    # Behaviour
    df["total_guests"]      = df["adults"] + df["children"] + df["babies"]
    df["total_stay_value"]  = df["adr"] * df["total_nights"]
    df["has_special_request"]= (df["total_of_special_requests"] > 0).astype(int)
    df["needs_parking"]      = (df["required_car_parking_spaces"] > 0).astype(int)
    df["room_was_changed"]   = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)
    df["has_cancel_history"] = (df["previous_cancellations"] > 0).astype(int)

    df["lead_time_bucket"] = pd.cut(df["lead_time"],
        bins=[0,7,30,90,180,365,9999],
        labels=["0-7d","8-30d","31-90d","91-180d","181-365d","365d+"])
    df["adr_tier"] = pd.cut(df["adr"],
        bins=[0,60,120,200,300,9999],
        labels=["Budget","Midrange","Upscale","Luxury","Ultra"])

    # Encode
    ohe_cols = ["hotel","meal","market_segment","distribution_channel",
                "deposit_type","customer_type","au_season"]
    label_cols = ["reserved_room_type","assigned_room_type","lead_time_bucket","adr_tier"]
    drop_cols  = ["arrival_date","arrival_date_month","country","agent","month_num"]

    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    df = pd.get_dummies(df, columns=[c for c in ohe_cols if c in df.columns], drop_first=True)
    le = LabelEncoder()
    for col in label_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    return df


if __name__ == "__main__":
    raw = pd.read_csv("././data/raw/hotel_bookings.csv")
    df_out = build_features(raw)
    X = df_out.drop(columns=["is_canceled"])
    y = df_out["is_canceled"]
    X.to_csv("././data/processed/X.csv", index=False)
    y.to_csv("././data/processed/y.csv", index=False)
    with open("././data/processed/feature_names.json","w") as f:
        json.dump(list(X.columns), f, indent=2)
    print(f"Done. X shape: {X.shape}")
'''

with open("./src/features/build_features.py", "w") as f:
    f.write(src_code)

print("Saved: src/features/build_features.py")

Saved: src/features/build_features.py
